# V18-C10: Simpson's Paradox Verification & All-Gene Exhaustive Analysis
## Subcluster Composition Confounding + 169-Gene Complete Screening

**Date:** 2026-03-05  
**Purpose:**  
1. **Subcluster composition analysis** — Verify whether lineage-level gene expression changes are driven by genuine per-cell changes vs. subcluster composition shifts (Simpson's Paradox at subcluster level)  
2. **All-gene exhaustive analysis** — Analyze ALL 169 genes (not top10) to avoid missing important 'minor' genes that may play critical roles  

**Principle:** No shortcuts in natural science. Bottom-up, data-driven. Every gene gets equal scrutiny.  

**Data:** `GSE182159_gut2021_annotated.h5ad` (243,000 cells, 23 donors, 59 subclusters)  
**Statistical unit:** Donor-level (Mann-Whitney U). Cell-level statistics = pseudo-replication = prohibited.  
**Tissue:** Liver and Blood analyzed SEPARATELY. Combined analysis absolutely prohibited.

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings, os, time, gc
warnings.filterwarnings('ignore')

print("✅ All libraries loaded")
print("=" * 70)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 134.4 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total
✅ All libraries loaded


---
## Cell 1: Setup & Data Loading

In [3]:
# ============================================================
# Cell 1: Setup & Data Loading
# ============================================================
import numpy as np
import pandas as pd
import scanpy as sc
import warnings
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['figure.facecolor'] = 'white'
warnings.filterwarnings('ignore')

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# MANDATORY DATA PATH — 'processed/' must never be omitted
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C10_SimpsonsParadox/'

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Load data
print('Loading h5ad...')
adata = sc.read_h5ad(DATA_PATH)
print(f'Loaded: {adata.shape[0]} cells × {adata.shape[1]} genes')
print(f'obs columns: {list(adata.obs.columns)}')

# Use 'major_lineage' based on obs columns
if 'major_lineage' in adata.obs.columns:
    print(f'Unique lineages: {adata.obs["major_lineage"].unique()}')
else:
    print("⚠️ 'major_lineage' column not found. Checking available columns...")

# Use 'gut2021_subcluster' based on obs columns
if 'gut2021_subcluster' in adata.obs.columns:
    print(f'Unique subclusters: {adata.obs["gut2021_subcluster"].nunique()}')

# Use 'sample' as likely donor column
if 'sample' in adata.obs.columns:
    print(f'Unique donors (samples): {adata.obs["sample"].nunique()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading h5ad...
Loaded: 243000 cells × 24452 genes
obs columns: ['sample', 'tissue', 'Stage', 'IT_cluster_21', 'IT_cluster_23', 'IT_cluster_25', 'IT_nk_collapse', 'IT_IT_signature', 'GSM_ID', 'IT_score_v2', 'IT_score_v3', 'IT_score_v4', 'IT_signature_final', 'IT_like', 'PW_mTOR_signaling', 'PW_glycolysis', 'PW_oxidative_phosphorylation', 'PW_nk_cell_cytotoxicity', 'PW_il15_signaling', 'PW_b_cell_differentiation', 'leiden', 'gut2021_subcluster', 'major_lineage', 'gut2021_subcluster_v2', 'TCR_clone.id', 'TCR_v_gene.x', 'TCR_j_gene.x', 'TCR_cdr3_nt.x', 'TCR_CType', 'BCR_clone.id', 'BCR_v_gene', 'BCR_j_gene', 'BCR_cdr3_nt', 'BCR_CType']
Unique lineages: ['B', 'CD8_T', 'Myeloid', 'NK', 'CD4_T', 'PlasmaB', 'gdT']
Categories (7, object): ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']
Unique subclusters: 48
Unique donors (samples): 46


---
## Cell 2: Identify Column Names & Define Constants
Adapt column names based on actual h5ad metadata structure.

In [10]:
# ============================================================
# Cell 2: Identify column names & constants
# ============================================================

# !! ADJUSTED COLUMN NAMES based on dataset structure !!
COL_LINEAGE = 'major_lineage'
COL_SUBCLUSTER = 'gut2021_subcluster'
COL_TISSUE = 'tissue'
COL_DONOR = 'sample'
COL_DISEASE = 'Stage'

# Validate
for col_name, col_val in [('LINEAGE', COL_LINEAGE), ('SUBCLUSTER', COL_SUBCLUSTER),
                           ('TISSUE', COL_TISSUE), ('DONOR', COL_DONOR), ('DISEASE', COL_DISEASE)]:
    assert col_val in adata.obs.columns, f'{col_name} column "{col_val}" not found!'
    print(f'✅ {col_name}: {col_val} → {adata.obs[col_val].nunique()} unique values')

# Disease group definitions
DISEASE_GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']
TISSUES = ['Liver', 'Blood']
# UPDATED: Matched lineage names to dataset (CD4_T, CD8_T)
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']

# Verify disease groups exist in data
actual_groups = adata.obs[COL_DISEASE].unique()
print(f'\nActual disease groups in data: {sorted(actual_groups)}')
print(f'Expected: {DISEASE_GROUPS}')

# Verify tissues
actual_tissues = adata.obs[COL_TISSUE].unique()
print(f'Actual tissues: {sorted(actual_tissues)}')

# Verify lineages
actual_lineages = adata.obs[COL_LINEAGE].unique()
print(f'Actual lineages: {sorted(actual_lineages)}')

# ============================================================
# PROACTIVE FIX: Handle KeyError: None / NaNs in Metadata
# ============================================================
print("\n=== Proactive Data Cleaning: Removing NaNs/None ===")
critical_cols = [COL_LINEAGE, COL_SUBCLUSTER, COL_TISSUE, COL_DONOR, COL_DISEASE]
cleaned_count = 0

for col in critical_cols:
    if col not in adata.obs.columns:
        continue

    # 1. Convert to string to avoid mixed type issues
    if adata.obs[col].dtype.name == 'category':
         # Handle categories by adding 'Unknown' if needed
         if '' not in adata.obs[col].cat.categories:
             pass # Category is fine usually, but NaNs need filling
    else:
        adata.obs[col] = adata.obs[col].astype(str)

    # 2. Fill NaNs
    if adata.obs[col].isna().any():
        print(f"⚠️ Found NaNs in '{col}'. Filling with 'Unknown'...")
        # For categorical, we must add the category first
        if isinstance(adata.obs[col].dtype, pd.CategoricalDtype):
            adata.obs[col] = adata.obs[col].cat.add_categories(['Unknown']).fillna('Unknown')
        else:
            adata.obs[col] = adata.obs[col].fillna('Unknown')
        cleaned_count += 1

    # 3. Check for string 'None' or 'nan'
    # Convert to string for checking
    temp_series = adata.obs[col].astype(str)
    if (temp_series == 'None').any() or (temp_series == 'nan').any():
        print(f"⚠️ Found 'None'/'nan' strings in '{col}'. Replacing...")
        mask = (temp_series == 'None') | (temp_series == 'nan')

        if isinstance(adata.obs[col].dtype, pd.CategoricalDtype):
             if 'Unknown' not in adata.obs[col].cat.categories:
                 adata.obs[col] = adata.obs[col].cat.add_categories(['Unknown'])
             adata.obs.loc[mask, col] = 'Unknown'
        else:
             adata.obs.loc[mask, col] = 'Unknown'
        cleaned_count += 1

if cleaned_count == 0:
    print("✅ No NaNs or 'None' values found in critical metadata.")
else:
    print(f"✅ Cleaned issues in {cleaned_count} columns.")

✅ LINEAGE: major_lineage → 7 unique values
✅ SUBCLUSTER: gut2021_subcluster → 48 unique values
✅ TISSUE: tissue → 2 unique values
✅ DONOR: sample → 46 unique values
✅ DISEASE: Stage → 5 unique values

Actual disease groups in data: ['AR', 'CR', 'IA', 'IT', 'NL']
Expected: ['NL', 'IT', 'IA', 'AR', 'CR']
Actual tissues: ['Blood', 'Liver']
Actual lineages: ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']

=== Proactive Data Cleaning: Removing NaNs/None ===
✅ No NaNs or 'None' values found in critical metadata.


---
## Cell 3: Define ALL 169 Gene Candidates
No top-10 filtering. Every gene that was analyzed in C3/C5 gets included.

In [11]:
# ============================================================
# Cell 3: Define ALL 169 gene candidates
# ============================================================
# Complete 169-gene list from C3 analysis
# Organized by functional category but ALL included in analysis

GENES_169 = [
    # Antigen Presentation (9)
    'HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'HLA-DQB1',
    'CD74', 'B2M', 'TAP1', 'TAP2',
    # JAK-STAT / Signaling (12)
    'JAK1', 'JAK2', 'JAK3', 'STAT1', 'STAT2', 'STAT3', 'STAT5A',
    'TYK2', 'SOCS1', 'SOCS3', 'CISH', 'PTPN1',
    # mTOR / PI3K (8)
    'MTOR', 'RPTOR', 'RICTOR', 'TSC1', 'TSC2', 'AKT1', 'PIK3CA', 'PIK3CD',
    # TGF-beta (6)
    'TGFB1', 'TGFBR1', 'TGFBR2', 'SMAD2', 'SMAD3', 'SMAD7',
    # Epigenetics (8)
    'DNMT1', 'DNMT3A', 'DNMT3B', 'TET2', 'EZH2', 'KDM6A', 'HDAC1', 'HDAC2',
    # Exhaustion / Checkpoint (14)
    'TOX', 'TOX2', 'LAYN', 'CTLA4', 'TIGIT', 'PDCD1', 'LAG3',
    'HAVCR2', 'BTLA', 'CD244', 'ENTPD1', 'CD160', 'KLRG1', 'EOMES',
    # Transcription Factors (12)
    'TBX21', 'GATA3', 'RORC', 'FOXP3', 'BCL6', 'PRDM1',
    'IRF4', 'IRF8', 'BATF', 'MAF', 'ID2', 'ID3',
    # Cytotoxicity / Effector (14)
    'PRF1', 'GZMA', 'GZMB', 'GZMK', 'GZMH', 'GNLY',
    'FASLG', 'IFNG', 'TNF', 'IL2', 'IL21', 'CSF2',
    'NKG7', 'KLRD1',
    # Apoptosis (8)
    'BAK1', 'BAX', 'BCL2', 'MCL1', 'CASP3', 'CASP8', 'FAS', 'TP53',
    # Inflammasome (8)
    'AIM2', 'NLRP3', 'NLRC4', 'MEFV', 'PYCARD', 'CASP1', 'IL1B', 'IL18',
    # Metabolic (12)
    'TFAM', 'LDHA', 'HIF1A', 'NDUFS1', 'SDHB', 'PKM',
    'MT-CYB', 'MT-ND1', 'MT-ND2', 'MT-CO1', 'MT-CO2', 'MT-ATP6',
    # Immune Regulation (10)
    'LGALS9', 'LILRB1', 'SIGLEC10', 'CD47', 'SIRPA',
    'IL10', 'IL10RA', 'IL1RN', 'IDO1', 'ARG1',
    # Cell Surface / Activation (14)
    'CD27', 'CD28', 'ICOS', 'IL2RA', 'IL7R', 'CD69',
    'SELL', 'CCR7', 'NCAM1', 'ITGAE', 'CD44',
    'TYROBP', 'FCER1G', 'FCGR3A',
    # B Cell Specific (8)
    'AICDA', 'JCHAIN', 'XBP1', 'IRF1', 'TNFRSF13B',
    'PAX5', 'CD19', 'MS4A1',
    # Fibrosis / Tissue (6)
    'COL1A1', 'FN1', 'SERPINE1', 'TIMP1', 'MMP9', 'IGFBP7',
    # IFN Response (8)
    'MX1', 'ISG15', 'IRF3', 'IRF7', 'IFNAR1', 'OAS1', 'IFI44L', 'IFIT1',
    # Cell Cycle / Proliferation (6)
    'MKI67', 'CDK4', 'CDK6', 'CCND1', 'CCNB1', 'TOP2A',
    # NK Specific (4)
    'KLRF1', 'NCR1', 'NCR3', 'KIR2DL1',
    # Additional from 6-layer model (6)
    'ATM', 'AXIN1', 'CDK1', 'CDKN1A', 'RB1', 'MDM2',
]

# Remove duplicates, preserve order
seen = set()
GENES_169 = [g for g in GENES_169 if g not in seen and not seen.add(g)]

# Check which genes exist in the dataset
available_genes = [g for g in GENES_169 if g in adata.var_names]
missing_genes = [g for g in GENES_169 if g not in adata.var_names]

print(f'Total candidate genes: {len(GENES_169)}')
print(f'Available in dataset: {len(available_genes)}')
print(f'Missing from dataset: {len(missing_genes)}')
if missing_genes:
    print(f'Missing genes: {missing_genes}')

GENES = available_genes  # Use all available genes
print(f'\n==> Will analyze {len(GENES)} genes across all lineages and tissues')

Total candidate genes: 173
Available in dataset: 173
Missing from dataset: 0

==> Will analyze 173 genes across all lineages and tissues


---
## PART A: Subcluster Composition Analysis
### Cell 4: Subcluster Proportion Table (59 subclusters × 5 groups × 2 tissues)

In [12]:
# ============================================================
# Cell 4: Subcluster proportion table — donor-level
# For each donor × tissue: proportion of each subcluster within its lineage
# ============================================================

def compute_subcluster_proportions(adata, tissue):
    """
    For each donor in a given tissue:
      For each lineage:
        Compute proportion of each subcluster within that lineage
    Returns: DataFrame with donor × subcluster proportions
    """
    mask = adata.obs[COL_TISSUE] == tissue
    sub = adata.obs[mask]

    records = []
    for donor in sub[COL_DONOR].unique():
        d_mask = sub[COL_DONOR] == donor
        disease = sub.loc[d_mask, COL_DISEASE].iloc[0]

        for lineage in sub[COL_LINEAGE].unique():
            dl_mask = d_mask & (sub[COL_LINEAGE] == lineage)
            n_lineage = dl_mask.sum()
            if n_lineage == 0:
                continue

            subclusters = sub.loc[dl_mask, COL_SUBCLUSTER].value_counts()
            for sc_name, sc_count in subclusters.items():
                records.append({
                    'donor': donor,
                    'disease': disease,
                    'tissue': tissue,
                    'lineage': lineage,
                    'subcluster': sc_name,
                    'n_cells_subcluster': sc_count,
                    'n_cells_lineage': n_lineage,
                    'proportion': sc_count / n_lineage
                })
    return pd.DataFrame(records)

print('Computing subcluster proportions...')
dfs = []
for tissue in TISSUES:
    df = compute_subcluster_proportions(adata, tissue)
    dfs.append(df)
    print(f'  {tissue}: {len(df)} records')

sc_prop_df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal records: {len(sc_prop_df)}')
print(f'Unique subclusters represented: {sc_prop_df["subcluster"].nunique()}')

# Save
sc_prop_df.to_csv(f'{RESULTS_DIR}C10_subcluster_proportions_donor_level.csv', index=False)
print(f'Saved to {RESULTS_DIR}')

Computing subcluster proportions...
  Liver: 7584 records
  Blood: 7584 records

Total records: 15168
Unique subclusters represented: 48
Saved to /content/drive/MyDrive/ITLAS/results/version18-analysis/C10_SimpsonsParadox/


### Cell 5: Statistical Test — Which Subclusters Change Significantly at NL→IT?

In [13]:
# ============================================================
# Cell 5: Subcluster proportion changes NL→IT (donor-level Mann-Whitney)
# ============================================================

def test_subcluster_proportion_changes(sc_prop_df, group1='NL', group2='IT'):
    """
    For each tissue × lineage × subcluster:
      Compare proportion in group1 vs group2 donors using Mann-Whitney U
    """
    results = []

    for tissue in TISSUES:
        t_df = sc_prop_df[sc_prop_df['tissue'] == tissue]

        for lineage in t_df['lineage'].unique():
            tl_df = t_df[t_df['lineage'] == lineage]

            # Get all subclusters within this lineage
            subclusters = tl_df['subcluster'].unique()

            # Get donors per group
            donors_g1 = tl_df[tl_df['disease'] == group1]['donor'].unique()
            donors_g2 = tl_df[tl_df['disease'] == group2]['donor'].unique()

            for sc_name in subclusters:
                # Get proportions for each donor (0 if subcluster absent)
                vals_g1 = []
                for d in donors_g1:
                    row = tl_df[(tl_df['donor'] == d) & (tl_df['subcluster'] == sc_name)]
                    vals_g1.append(row['proportion'].values[0] if len(row) > 0 else 0.0)

                vals_g2 = []
                for d in donors_g2:
                    row = tl_df[(tl_df['donor'] == d) & (tl_df['subcluster'] == sc_name)]
                    vals_g2.append(row['proportion'].values[0] if len(row) > 0 else 0.0)

                vals_g1 = np.array(vals_g1)
                vals_g2 = np.array(vals_g2)

                mean_g1 = np.mean(vals_g1)
                mean_g2 = np.mean(vals_g2)

                # Skip if both are essentially zero
                if mean_g1 < 0.001 and mean_g2 < 0.001:
                    continue

                # Mann-Whitney U (donor-level)
                try:
                    stat, pval = mannwhitneyu(vals_g1, vals_g2, alternative='two-sided')
                except:
                    pval = 1.0

                # Percent change
                if mean_g1 > 0:
                    pct_change = ((mean_g2 - mean_g1) / mean_g1) * 100
                else:
                    pct_change = 99999 if mean_g2 > 0 else 0

                results.append({
                    'tissue': tissue,
                    'lineage': lineage,
                    'subcluster': sc_name,
                    f'mean_{group1}': mean_g1,
                    f'mean_{group2}': mean_g2,
                    'pct_change': pct_change,
                    'p_value': pval,
                    f'n_{group1}': len(vals_g1),
                    f'n_{group2}': len(vals_g2)
                })

    df = pd.DataFrame(results)

    # FDR correction within tissue×lineage
    df['fdr_q'] = 1.0
    for (tissue, lineage), grp in df.groupby(['tissue', 'lineage']):
        if len(grp) > 1:
            _, qvals, _, _ = multipletests(grp['p_value'], method='fdr_bh')
            df.loc[grp.index, 'fdr_q'] = qvals
        else:
            df.loc[grp.index, 'fdr_q'] = grp['p_value'].values

    return df.sort_values(['tissue', 'lineage', 'p_value'])

# Run for all key comparisons
comparisons = [('NL', 'IT'), ('NL', 'IA'), ('IT', 'IA'), ('NL', 'CR'), ('NL', 'AR'), ('IA', 'AR')]
sc_change_dfs = {}

for g1, g2 in comparisons:
    print(f'Testing {g1}→{g2} subcluster proportion changes...')
    df = test_subcluster_proportion_changes(sc_prop_df, g1, g2)
    sc_change_dfs[f'{g1}_{g2}'] = df
    sig = df[df['p_value'] < 0.05]
    print(f'  {len(sig)} significant subcluster proportion changes (p<0.05)')
    if len(sig) > 0:
        print(sig[['tissue','lineage','subcluster','pct_change','p_value']].head(15).to_string())
    print()

# Save NL→IT results (most critical)
sc_change_dfs['NL_IT'].to_csv(f'{RESULTS_DIR}C10_subcluster_proportion_NLvsIT.csv', index=False)
print('\n==> Subcluster proportion changes saved.')

Testing NL→IT subcluster proportion changes...
  10 significant subcluster proportion changes (p<0.05)
    tissue  lineage        subcluster    pct_change   p_value
98   Blood    CD8_T    g9d2T-c01-CD27    -76.270816  0.047280
100  Blood  Myeloid  plasmaB_c02-CD52     42.631576  0.011682
101  Blood  Myeloid     mono_c01-CD14    -72.912773  0.011682
111  Blood       NK   CD8T_c03-CX3CR1   2415.772532  0.025631
56   Liver        B      cDC2-CLEC10A  99999.000000  0.048406
45   Liver  Myeloid     mono_c01-CD14    -92.533101  0.004772
42   Liver  Myeloid      cDC2-CLEC10A    378.920082  0.029480
30   Liver       NK     CD4T_c08-GZMK    -56.874520  0.004329
26   Liver       NK  plasmaB_c02-CD52     -9.078348  0.041126
75   Liver      gdT    CD4T_c07-TIMP1    -78.555911  0.027486

Testing NL→IA subcluster proportion changes...
  2 significant subcluster proportion changes (p<0.05)
   tissue  lineage          subcluster   pct_change   p_value
51  Liver        B         B_c05-TCL1A  1499.12616

---
## PART B: Decomposition — Composition vs. Per-Cell Expression
### Cell 6: For each gene, decompose lineage-level change into:
- (A) **Composition effect**: change due to subcluster proportion shifts
- (B) **Per-cell effect**: change within the same subclusters
- (C) **Interaction**: joint effect

In [14]:
# ============================================================
# Cell 6: Expression Decomposition — ALL genes, ALL lineages
# For each gene × lineage × tissue:
#   Total change = Composition effect + Per-cell effect + Interaction
#
# Method (Oaxaca-Blinder-like decomposition):
#   E[X|group2] - E[X|group1] =
#     Σ_s (π2_s - π1_s) × x̄1_s   [composition effect: subcluster weight change]
#   + Σ_s π1_s × (x̄2_s - x̄1_s)   [per-cell effect: within-subcluster expression change]
#   + Σ_s (π2_s - π1_s)(x̄2_s - x̄1_s)  [interaction term]
#
# Where: π_s = proportion of subcluster s within lineage
#        x̄_s = mean expression of gene in subcluster s
# ============================================================

def decompose_expression_change(adata, gene, tissue, lineage, group1='NL', group2='IT'):
    """
    Decompose lineage-level expression change into composition vs per-cell effects.
    Returns dict with decomposition results.
    Uses DONOR-LEVEL means to avoid pseudo-replication.
    """
    # Filter to tissue × lineage
    mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_LINEAGE] == lineage)
    sub = adata[mask]

    if gene not in sub.var_names:
        return None

    # Get gene expression (handle sparse)
    gene_idx = list(sub.var_names).index(gene)
    expr = sub.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()

    obs = sub.obs.copy()
    obs['expr'] = expr

    # Split by group
    g1_obs = obs[obs[COL_DISEASE] == group1]
    g2_obs = obs[obs[COL_DISEASE] == group2]

    if len(g1_obs) == 0 or len(g2_obs) == 0:
        return None

    # Get all subclusters present in either group
    all_sc = set(g1_obs[COL_SUBCLUSTER].unique()) | set(g2_obs[COL_SUBCLUSTER].unique())

    # Compute DONOR-LEVEL subcluster proportions and mean expression
    # Step 1: For each donor, get subcluster proportion within lineage
    # Step 2: Average across donors in each group

    def get_donor_level_stats(group_obs):
        """Get group-averaged subcluster proportions and expressions via donor-level means."""
        donors = group_obs[COL_DONOR].unique()
        donor_props = {}
        donor_exprs = {}

        for donor in donors:
            d_obs = group_obs[group_obs[COL_DONOR] == donor]
            n_total = len(d_obs)

            props = {}
            exprs = {}
            for sc_name in all_sc:
                sc_obs = d_obs[d_obs[COL_SUBCLUSTER] == sc_name]
                props[sc_name] = len(sc_obs) / n_total if n_total > 0 else 0
                exprs[sc_name] = sc_obs['expr'].mean() if len(sc_obs) > 0 else 0

            donor_props[donor] = props
            donor_exprs[donor] = exprs

        # Average across donors (donor-level mean, not cell-level)
        avg_props = {sc: np.mean([donor_props[d][sc] for d in donors]) for sc in all_sc}
        avg_exprs = {sc: np.mean([donor_exprs[d][sc] for d in donors]) for sc in all_sc}

        return avg_props, avg_exprs, donors

    pi1, x1, donors_g1 = get_donor_level_stats(g1_obs)
    pi2, x2, donors_g2 = get_donor_level_stats(g2_obs)

    # Decomposition
    composition_effect = 0
    percell_effect = 0
    interaction_effect = 0

    for sc_name in all_sc:
        d_pi = pi2[sc_name] - pi1[sc_name]
        d_x = x2[sc_name] - x1[sc_name]

        composition_effect += d_pi * x1[sc_name]
        percell_effect += pi1[sc_name] * d_x
        interaction_effect += d_pi * d_x

    total_change = composition_effect + percell_effect + interaction_effect

    # Also compute lineage-level donor means for Mann-Whitney
    donor_means_g1 = g1_obs.groupby(COL_DONOR)['expr'].mean().values
    donor_means_g2 = g2_obs.groupby(COL_DONOR)['expr'].mean().values

    lineage_mean_g1 = np.mean(donor_means_g1)
    lineage_mean_g2 = np.mean(donor_means_g2)

    try:
        _, pval = mannwhitneyu(donor_means_g1, donor_means_g2, alternative='two-sided')
    except:
        pval = 1.0

    if lineage_mean_g1 > 0:
        pct_change = ((lineage_mean_g2 - lineage_mean_g1) / lineage_mean_g1) * 100
    else:
        pct_change = 99999 if lineage_mean_g2 > 0 else 0

    # Proportion attribution (avoid div by zero)
    abs_total = abs(composition_effect) + abs(percell_effect) + abs(interaction_effect)

    return {
        'gene': gene,
        'tissue': tissue,
        'lineage': lineage,
        'comparison': f'{group1}→{group2}',
        'lineage_mean_g1': lineage_mean_g1,
        'lineage_mean_g2': lineage_mean_g2,
        'pct_change': pct_change,
        'p_value': pval,
        'n_donors_g1': len(donors_g1),
        'n_donors_g2': len(donors_g2),
        'total_change': total_change,
        'composition_effect': composition_effect,
        'percell_effect': percell_effect,
        'interaction_effect': interaction_effect,
        'pct_composition': (abs(composition_effect) / abs_total * 100) if abs_total > 0 else 0,
        'pct_percell': (abs(percell_effect) / abs_total * 100) if abs_total > 0 else 0,
        'pct_interaction': (abs(interaction_effect) / abs_total * 100) if abs_total > 0 else 0,
        'dominant_driver': 'composition' if abs(composition_effect) > abs(percell_effect) else 'per-cell',
        'n_subclusters': len(all_sc)
    }

print('Decomposition function defined. Ready for Cell 7.')

Decomposition function defined. Ready for Cell 7.


### Cell 7: Run Decomposition for ALL Genes × ALL Lineages × ALL Tissues
**This is the exhaustive analysis — no top-10 filtering.**

In [17]:
# === DIAGNOSTIC: 컬럼값 확인 ===
print("=== Disease groups ===")
print(adata.obs[COL_DISEASE].value_counts())
print("\n=== Tissues ===")
print(adata.obs[COL_TISSUE].value_counts())
print("\n=== Lineages ===")
print(adata.obs[COL_LINEAGE].value_counts())
print("\n=== Sample filter test ===")
for tissue in ['Liver', 'Blood', 'liver', 'blood', 'PBMC', 'Intrahepatic']:
    n = (adata.obs[COL_TISSUE] == tissue).sum()
    if n > 0:
        print(f"  tissue='{tissue}': {n} cells")

for disease in ['NL', 'IT', 'HC', 'Healthy', 'IT_phase', 'Immune Tolerant']:
    n = (adata.obs[COL_DISEASE] == disease).sum()
    if n > 0:
        print(f"  disease='{disease}': {n} cells")

=== Disease groups ===
Stage
IA    62545
IT    49179
AR    45452
CR    43245
NL    42579
Name: count, dtype: int64

=== Tissues ===
tissue
Blood    136408
Liver    106592
Name: count, dtype: int64

=== Lineages ===
major_lineage
CD8_T      70065
CD4_T      69446
NK         54584
Myeloid    24716
B          21325
PlasmaB     2083
gdT          781
Name: count, dtype: int64

=== Sample filter test ===
  tissue='Liver': 106592 cells
  tissue='Blood': 136408 cells
  disease='NL': 42579 cells
  disease='IT': 49179 cells


In [18]:
# === FIX: 컬럼명 수정 + donor/subcluster 확인 ===
COL_DISEASE = 'Stage'
COL_LINEAGE = 'major_lineage'
COL_TISSUE = 'tissue'

# donor & subcluster 컬럼 찾기
for col in adata.obs.columns:
    u = adata.obs[col].nunique()
    if 20 <= u <= 30:
        print(f"  Likely DONOR: '{col}' → {u} unique")
        print(f"    {adata.obs[col].unique()[:8]}")
    if 50 <= u <= 70:
        print(f"  Likely SUBCLUSTER: '{col}' → {u} unique")
        print(f"    {adata.obs[col].unique()[:8]}")

# Lineage값 확인
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
print(f"\nLINEAGES set to: {LINEAGES}")

  Likely DONOR: 'leiden' → 24 unique
    [ 7 12 17  6  4  0  3 21]
  Likely SUBCLUSTER: 'gut2021_subcluster_v2' → 59 unique
    ['B_c01-IGHD', 'CD8T_c01-LEF1', 'mono_c02-FCGR3A', 'NK_c03-PTGDS', 'CD8T_c03-CX3CR1', 'CD8T_c09-ACTN1', 'B_c03-CD1C', 'CD4T_c06-IFI44L']
Categories (59, object): ['B_c01-IGHD', 'B_c02-STX16', 'B_c03-CD1C', 'B_c04-COCH', ..., 'pDC',
                          'plasmaB_c01-SDC1', 'plasmaB_c02-CD52', 'plasmaB_c03-MKI67']
  Likely SUBCLUSTER: 'TCR_j_gene.x' → 53 unique
    [NaN, 'TRAJ58', 'TRAJ34', 'TRAJ42', 'TRAJ8', 'TRAJ22', 'TRAJ16', 'TRAJ12']
Categories (53, object): ['TRAJ3', 'TRAJ4', 'TRAJ5', 'TRAJ6', ..., 'TRAJ56', 'TRAJ57', 'TRAJ58',
                          'TRAJ61']
  Likely SUBCLUSTER: 'BCR_v_gene' → 50 unique
    ['IGHV1-69D', NaN, 'IGHV4-59', 'IGHV3-13', 'IGHV4-39', 'IGHV3-33', 'IGHV3-66', 'IGHV1-46']
Categories (50, object): ['IGHV1-2', 'IGHV1-3', 'IGHV1-8', 'IGHV1-18', ..., 'IGHV5-51', 'IGHV6-1',
                          'IGHV7-4-1', 'IGHV7-81']

L

In [19]:
# === donor 컬럼 정확히 찾기 ===
print("=== 23 donors 찾기 (nunique 20~25 범위, 문자열) ===")
for col in adata.obs.columns:
    u = adata.obs[col].nunique()
    if 15 <= u <= 30:
        sample = adata.obs[col].unique()[:5]
        dtype = adata.obs[col].dtype
        print(f"  '{col}': {u} unique, dtype={dtype}, sample={sample}")

print("\n=== 모든 obs 컬럼 (nunique 포함) ===")
for col in adata.obs.columns:
    print(f"  {col}: {adata.obs[col].nunique()} unique")

=== 23 donors 찾기 (nunique 20~25 범위, 문자열) ===
  'leiden': 24 unique, dtype=int64, sample=[ 7 12 17  6  4]

=== 모든 obs 컬럼 (nunique 포함) ===
  sample: 46 unique
  tissue: 2 unique
  Stage: 5 unique
  IT_cluster_21: 242960 unique
  IT_cluster_23: 240984 unique
  IT_cluster_25: 114299 unique
  IT_nk_collapse: 168875 unique
  IT_IT_signature: 243000 unique
  GSM_ID: 46 unique
  IT_score_v2: 243000 unique
  IT_score_v3: 243000 unique
  IT_score_v4: 242998 unique
  IT_signature_final: 242998 unique
  IT_like: 2 unique
  PW_mTOR_signaling: 30497 unique
  PW_glycolysis: 208005 unique
  PW_oxidative_phosphorylation: 237150 unique
  PW_nk_cell_cytotoxicity: 137083 unique
  PW_il15_signaling: 93432 unique
  PW_b_cell_differentiation: 57564 unique
  leiden: 24 unique
  gut2021_subcluster: 48 unique
  major_lineage: 7 unique
  gut2021_subcluster_v2: 59 unique
  TCR_clone.id: 40517 unique
  TCR_v_gene.x: 44 unique
  TCR_j_gene.x: 53 unique
  TCR_cdr3_nt.x: 36655 unique
  TCR_CType: 1 unique
  BCR_clone

In [21]:
# === sample → donor 매핑 확인 ===
# sample이 donor×tissue 조합인지 확인
cross = adata.obs.groupby(['sample', 'tissue', 'Stage']).size().reset_index(name='n')
print(cross.sort_values('sample').to_string())

# donor 추출 시도 (이전 코드 패턴)
print("\n=== sample 값 예시 ===")
print(sorted(adata.obs['sample'].unique()))

                         sample tissue Stage      n
0    GSM5519467_P190604_Blood_1  Blood    CR      0
1    GSM5519467_P190604_Blood_1  Blood    AR      0
2    GSM5519467_P190604_Blood_1  Blood    IA      0
3    GSM5519467_P190604_Blood_1  Blood    IT   1891
4    GSM5519467_P190604_Blood_1  Blood    NL      0
5    GSM5519467_P190604_Blood_1  Liver    CR      0
6    GSM5519467_P190604_Blood_1  Liver    AR      0
7    GSM5519467_P190604_Blood_1  Liver    IA      0
8    GSM5519467_P190604_Blood_1  Liver    IT      0
9    GSM5519467_P190604_Blood_1  Liver    NL      0
19   GSM5519468_P190604_Blood_2  Liver    NL      0
18   GSM5519468_P190604_Blood_2  Liver    IT      0
17   GSM5519468_P190604_Blood_2  Liver    IA      0
16   GSM5519468_P190604_Blood_2  Liver    AR      0
15   GSM5519468_P190604_Blood_2  Liver    CR      0
11   GSM5519468_P190604_Blood_2  Blood    AR      0
13   GSM5519468_P190604_Blood_2  Blood    IT   1784
12   GSM5519468_P190604_Blood_2  Blood    IA      0
10   GSM5519

In [22]:
# === COMPLETE FIX: 모든 변수 재설정 ===

# 1) 컬럼명 수정
COL_DISEASE = 'Stage'
COL_LINEAGE = 'major_lineage'
COL_TISSUE = 'tissue'
COL_SUBCLUSTER = 'gut2021_subcluster_v2'
COL_DONOR = 'donor_id'  # 새로 만들 컬럼

# 2) sample → donor_id 추출 (P190604, D528848, Dhc570 등)
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]

# 3) 검증
print("=== Donor ID 검증 ===")
print(f"Unique donors: {adata.obs['donor_id'].nunique()}")
print(sorted(adata.obs['donor_id'].unique()))

# donor × disease × tissue 크로스탭
cross = adata.obs.groupby(['donor_id', COL_DISEASE, COL_TISSUE]).size().reset_index(name='n')
cross = cross[cross['n'] > 0].sort_values(['donor_id', COL_TISSUE])
print(f"\n=== Donor × Disease × Tissue (n>0 only) ===")
print(cross.to_string())

# 4) Lineage 및 기타 상수
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']
TISSUES = ['Liver', 'Blood']
DISEASE_GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']

# 5) 최종 확인
print(f"\n✅ COL_DISEASE = '{COL_DISEASE}'")
print(f"✅ COL_LINEAGE = '{COL_LINEAGE}'")
print(f"✅ COL_TISSUE = '{COL_TISSUE}'")
print(f"✅ COL_SUBCLUSTER = '{COL_SUBCLUSTER}'")
print(f"✅ COL_DONOR = '{COL_DONOR}'")
print(f"✅ LINEAGES = {LINEAGES}")

# 6) 빠른 테스트: NL→IT Blood Myeloid에서 HLA-DRA
test_mask = (adata.obs[COL_TISSUE] == 'Blood') & (adata.obs[COL_LINEAGE] == 'Myeloid')
test_sub = adata[test_mask]
gene_idx = list(test_sub.var_names).index('HLA-DRA')
expr = test_sub.X[:, gene_idx]
if hasattr(expr, 'toarray'):
    expr = expr.toarray().flatten()
test_obs = test_sub.obs.copy()
test_obs['expr'] = expr
donor_means = test_obs.groupby([COL_DONOR, COL_DISEASE])['expr'].mean().reset_index()
nl = donor_means[donor_means[COL_DISEASE] == 'NL']['expr'].values
it = donor_means[donor_means[COL_DISEASE] == 'IT']['expr'].values
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(nl, it, alternative='two-sided')
print(f"\n=== Quick Test: HLA-DRA Blood Myeloid NL→IT ===")
print(f"NL donors: {len(nl)}, mean={nl.mean():.4f}")
print(f"IT donors: {len(it)}, mean={it.mean():.4f}")
print(f"Change: {((it.mean()-nl.mean())/nl.mean()*100):+.1f}%")
print(f"Mann-Whitney p = {p:.6f}")
print(f"{'✅ SIGNIFICANT' if p < 0.05 else '❌ NOT SIGNIFICANT'}")

=== Donor ID 검증 ===
Unique donors: 23
['D528848', 'D529074', 'D529351', 'D529354', 'D529409', 'Dhc570', 'P190326', 'P190402', 'P190604', 'P190716', 'P190719', 'P190801', 'P190808', 'P190902', 'P190910', 'P190911', 'P191008', 'P191028', 'P191112', 'P191126', 'P191127', 'P191210', 'P191217']

=== Donor × Disease × Tissue (n>0 only) ===
    donor_id Stage tissue      n
8    D528848    NL  Blood   8149
9    D528848    NL  Liver   4733
18   D529074    NL  Blood   6843
19   D529074    NL  Liver   5956
28   D529351    NL  Blood   1988
29   D529351    NL  Liver    690
38   D529354    NL  Blood    720
39   D529354    NL  Liver   5233
48   D529409    NL  Blood    660
49   D529409    NL  Liver   3364
59    Dhc570    NL  Liver   4243
66   P190326    IT  Blood   3674
67   P190326    IT  Liver   1379
77   P190402    IT  Liver   7110
86   P190604    IT  Blood   3675
87   P190604    IT  Liver   3085
92   P190716    AR  Blood   4916
93   P190716    AR  Liver   3460
104  P190719    IA  Blood   3934
105 

In [23]:
# === Sparse matrix 처리 수정 + 테스트 ===
import numpy as np
from scipy import sparse

# expr 추출 확인
test_mask = (adata.obs[COL_TISSUE] == 'Blood') & (adata.obs[COL_LINEAGE] == 'Myeloid')
test_sub = adata[test_mask]
print(f"Blood Myeloid cells: {test_sub.shape[0]}")
print(f"X type: {type(test_sub.X)}")

gene_idx = list(test_sub.var_names).index('HLA-DRA')

# sparse 안전 추출
if sparse.issparse(test_sub.X):
    expr = np.asarray(test_sub.X[:, gene_idx].todense()).flatten()
else:
    expr = np.array(test_sub.X[:, gene_idx]).flatten()

print(f"expr shape: {expr.shape}, NaN count: {np.isnan(expr).sum()}, mean: {np.nanmean(expr):.4f}")

# donor별 mean (해당 tissue+lineage에 실제 세포가 있는 donor만)
test_obs = test_sub.obs.copy()
test_obs['expr'] = expr

donor_means = test_obs.groupby([COL_DONOR, COL_DISEASE])['expr'].mean().reset_index()
donor_means = donor_means.dropna(subset=['expr'])

nl = donor_means[donor_means[COL_DISEASE] == 'NL']['expr'].values
it = donor_means[donor_means[COL_DISEASE] == 'IT']['expr'].values
print(f"\nNL: {len(nl)} donors, mean={nl.mean():.4f}")
print(f"IT: {len(it)} donors, mean={it.mean():.4f}")
print(f"Change: {((it.mean()-nl.mean())/nl.mean()*100):+.1f}%")

from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(nl, it, alternative='two-sided')
print(f"p = {p:.6f} {'✅ SIGNIFICANT' if p < 0.05 else '❌ NS'}")

Blood Myeloid cells: 21224
X type: <class 'anndata._core.views.ArrayView'>
expr shape: (21224,), NaN count: 0, mean: 2.9300

NL: 5 donors, mean=2.0579
IT: 5 donors, mean=3.2160
Change: +56.3%
p = 0.007937 ✅ SIGNIFICANT


In [25]:
# === Sparse matrix 처리 수정 + 테스트 ===
import numpy as np
from scipy import sparse

# expr 추출 확인
test_mask = (adata.obs[COL_TISSUE] == 'Blood') & (adata.obs[COL_LINEAGE] == 'Myeloid')
test_sub = adata[test_mask]
print(f"Blood Myeloid cells: {test_sub.shape[0]}")
print(f"X type: {type(test_sub.X)}")

gene_idx = list(test_sub.var_names).index('HLA-DRA')

# sparse 안전 추출
if sparse.issparse(test_sub.X):
    expr = np.asarray(test_sub.X[:, gene_idx].todense()).flatten()
else:
    expr = np.array(test_sub.X[:, gene_idx]).flatten()

print(f"expr shape: {expr.shape}, NaN count: {np.isnan(expr).sum()}, mean: {np.nanmean(expr):.4f}")

# donor별 mean (해당 tissue+lineage에 실제 세포가 있는 donor만)
test_obs = test_sub.obs.copy()
test_obs['expr'] = expr

donor_means = test_obs.groupby([COL_DONOR, COL_DISEASE])['expr'].mean().reset_index()
donor_means = donor_means.dropna(subset=['expr'])

nl = donor_means[donor_means[COL_DISEASE] == 'NL']['expr'].values
it = donor_means[donor_means[COL_DISEASE] == 'IT']['expr'].values
print(f"\nNL: {len(nl)} donors, mean={nl.mean():.4f}")
print(f"IT: {len(it)} donors, mean={it.mean():.4f}")
print(f"Change: {((it.mean()-nl.mean())/nl.mean()*100):+.1f}%")

from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(nl, it, alternative='two-sided')
print(f"p = {p:.6f} {'✅ SIGNIFICANT' if p < 0.05 else '❌ NS'}")

Blood Myeloid cells: 21224
X type: <class 'anndata._core.views.ArrayView'>
expr shape: (21224,), NaN count: 0, mean: 2.9300

NL: 5 donors, mean=2.0579
IT: 5 donors, mean=3.2160
Change: +56.3%
p = 0.007937 ✅ SIGNIFICANT


In [26]:
# ============================================================
# Cell 6 FIXED: Expression Decomposition — ALL genes, ALL lineages
# Sparse/Dense matrix 안전 처리 포함
# ============================================================
import numpy as np
from scipy import sparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

def safe_get_expr(adata_sub, gene):
    """Sparse/Dense 모두 안전하게 gene expression 추출"""
    gene_idx = list(adata_sub.var_names).index(gene)
    X = adata_sub.X
    if sparse.issparse(X):
        return np.asarray(X[:, gene_idx].todense()).flatten()
    else:
        return np.asarray(X[:, gene_idx]).flatten()

def decompose_expression_change(adata, gene, tissue, lineage, group1='NL', group2='IT'):
    """
    Decompose lineage-level expression change into composition vs per-cell effects.
    Oaxaca-Blinder decomposition, donor-level aggregation.
    """
    mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_LINEAGE] == lineage)
    sub = adata[mask]

    if gene not in sub.var_names:
        return None
    if sub.shape[0] < 10:
        return None

    expr = safe_get_expr(sub, gene)
    obs = sub.obs.copy()
    obs['expr'] = expr

    g1_obs = obs[obs[COL_DISEASE] == group1]
    g2_obs = obs[obs[COL_DISEASE] == group2]

    if len(g1_obs) == 0 or len(g2_obs) == 0:
        return None

    all_sc = set(g1_obs[COL_SUBCLUSTER].unique()) | set(g2_obs[COL_SUBCLUSTER].unique())

    def get_donor_level_stats(group_obs):
        donors = group_obs[COL_DONOR].unique()
        donor_props = {}
        donor_exprs = {}

        for donor in donors:
            d_obs = group_obs[group_obs[COL_DONOR] == donor]
            n_total = len(d_obs)

            props = {}
            exprs = {}
            for sc_name in all_sc:
                sc_obs = d_obs[d_obs[COL_SUBCLUSTER] == sc_name]
                props[sc_name] = len(sc_obs) / n_total if n_total > 0 else 0
                exprs[sc_name] = sc_obs['expr'].mean() if len(sc_obs) > 0 else 0

            donor_props[donor] = props
            donor_exprs[donor] = exprs

        avg_props = {sc: np.mean([donor_props[d][sc] for d in donors]) for sc in all_sc}
        avg_exprs = {sc: np.mean([donor_exprs[d][sc] for d in donors]) for sc in all_sc}

        return avg_props, avg_exprs, donors

    pi1, x1, donors_g1 = get_donor_level_stats(g1_obs)
    pi2, x2, donors_g2 = get_donor_level_stats(g2_obs)

    # Oaxaca-Blinder decomposition
    composition_effect = 0
    percell_effect = 0
    interaction_effect = 0

    for sc_name in all_sc:
        d_pi = pi2[sc_name] - pi1[sc_name]
        d_x = x2[sc_name] - x1[sc_name]
        composition_effect += d_pi * x1[sc_name]
        percell_effect += pi1[sc_name] * d_x
        interaction_effect += d_pi * d_x

    total_change = composition_effect + percell_effect + interaction_effect

    # Lineage-level donor means for Mann-Whitney
    donor_means_g1 = g1_obs.groupby(COL_DONOR)['expr'].mean().dropna().values
    donor_means_g2 = g2_obs.groupby(COL_DONOR)['expr'].mean().dropna().values

    if len(donor_means_g1) < 2 or len(donor_means_g2) < 2:
        return None

    lineage_mean_g1 = np.mean(donor_means_g1)
    lineage_mean_g2 = np.mean(donor_means_g2)

    try:
        _, pval = mannwhitneyu(donor_means_g1, donor_means_g2, alternative='two-sided')
    except:
        pval = 1.0

    if lineage_mean_g1 > 1e-10:
        pct_change = ((lineage_mean_g2 - lineage_mean_g1) / lineage_mean_g1) * 100
    else:
        pct_change = 99999 if lineage_mean_g2 > 1e-10 else 0

    abs_total = abs(composition_effect) + abs(percell_effect) + abs(interaction_effect)

    return {
        'gene': gene,
        'tissue': tissue,
        'lineage': lineage,
        'comparison': f'{group1}→{group2}',
        'lineage_mean_g1': lineage_mean_g1,
        'lineage_mean_g2': lineage_mean_g2,
        'pct_change': pct_change,
        'p_value': pval,
        'n_donors_g1': len(donors_g1),
        'n_donors_g2': len(donors_g2),
        'total_change': total_change,
        'composition_effect': composition_effect,
        'percell_effect': percell_effect,
        'interaction_effect': interaction_effect,
        'pct_composition': (abs(composition_effect) / abs_total * 100) if abs_total > 1e-15 else 0,
        'pct_percell': (abs(percell_effect) / abs_total * 100) if abs_total > 1e-15 else 0,
        'pct_interaction': (abs(interaction_effect) / abs_total * 100) if abs_total > 1e-15 else 0,
        'dominant_driver': 'composition' if abs(composition_effect) > abs(percell_effect) else 'per-cell',
        'n_subclusters': len(all_sc)
    }

# === Quick validation ===
test = decompose_expression_change(adata, 'HLA-DRA', 'Blood', 'Myeloid', 'NL', 'IT')
if test:
    print(f"✅ HLA-DRA Blood Myeloid: {test['pct_change']:+.1f}% p={test['p_value']:.4f}")
    print(f"   Composition: {test['pct_composition']:.1f}%")
    print(f"   Per-cell:    {test['pct_percell']:.1f}%")
    print(f"   Interaction: {test['pct_interaction']:.1f}%")
    print(f"   Driver: {test['dominant_driver']}")
else:
    print("❌ Test failed")

# Test a few more key genes
for gene, tissue, lineage in [('TOX', 'Liver', 'CD4_T'), ('DNMT1', 'Blood', 'Myeloid'),
                                ('SOCS1', 'Blood', 'CD4_T'), ('PRDM1', 'Liver', 'CD4_T')]:
    r = decompose_expression_change(adata, gene, tissue, lineage, 'NL', 'IT')
    if r:
        flag = '⚠️COMP' if r['pct_composition'] > 50 else '✅CELL'
        print(f"  {gene} {tissue}/{lineage}: {r['pct_change']:+.1f}% p={r['p_value']:.4f} "
              f"comp={r['pct_composition']:.0f}% cell={r['pct_percell']:.0f}% {flag}")

✅ HLA-DRA Blood Myeloid: +56.3% p=0.0079
   Composition: 1.0%
   Per-cell:    94.0%
   Interaction: 5.0%
   Driver: per-cell
  TOX Liver/CD4_T: +100.7% p=0.0087 comp=9% cell=73% ✅CELL
  DNMT1 Blood/Myeloid: +163.8% p=0.0079 comp=3% cell=96% ✅CELL
  SOCS1 Blood/CD4_T: -67.0% p=0.0079 comp=7% cell=87% ✅CELL
  PRDM1 Liver/CD4_T: -36.4% p=0.0022 comp=1% cell=98% ✅CELL


In [27]:
# ============================================================
# Cell 7: Exhaustive decomposition — ALL genes × ALL lineages × ALL tissues
# ALL comparisons: NL→IT, NL→IA, IT→IA, NL→CR, NL→AR, IA→AR
# ============================================================

all_decomp_results = []
total_tests = len(GENES) * len(LINEAGES) * len(TISSUES) * len(comparisons)
print(f'Total tests to run: {len(GENES)} genes × {len(LINEAGES)} lineages × {len(TISSUES)} tissues × {len(comparisons)} comparisons = {total_tests}')
print(f'This may take 15-30 minutes on A100...')
print()

counter = 0
for comp_idx, (g1, g2) in enumerate(comparisons):
    print(f'\n=== Comparison {comp_idx+1}/{len(comparisons)}: {g1}→{g2} ===')
    comp_results = []

    for tissue in TISSUES:
        for lineage in LINEAGES:
            # Check if this lineage exists in this tissue
            mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_LINEAGE] == lineage)
            if mask.sum() < 10:
                continue

            for gene in GENES:
                result = decompose_expression_change(adata, gene, tissue, lineage, g1, g2)
                if result is not None:
                    comp_results.append(result)

                counter += 1
                if counter % 2000 == 0:
                    print(f'  Progress: {counter}/{total_tests} ({counter/total_tests*100:.1f}%)')

    if not comp_results:
        print(f'  No results for comparison {g1}→{g2}')
        continue

    comp_df = pd.DataFrame(comp_results)

    # FDR correction within tissue×lineage
    comp_df['fdr_q'] = 1.0
    for (tissue, lineage), grp in comp_df.groupby(['tissue', 'lineage']):
        if len(grp) > 1:
            _, qvals, _, _ = multipletests(grp['p_value'], method='fdr_bh')
            comp_df.loc[grp.index, 'fdr_q'] = qvals

    all_decomp_results.append(comp_df)

    # Summary for this comparison
    sig = comp_df[comp_df['p_value'] < 0.05]
    comp_driven = sig[sig['dominant_driver'] == 'composition']
    percell_driven = sig[sig['dominant_driver'] == 'per-cell']
    print(f'  Significant (p<0.05): {len(sig)}')

    # FIX: Check if len(sig) > 0 before division to avoid ZeroDivisionError
    if len(sig) > 0:
        print(f'  → Composition-dominant: {len(comp_driven)} ({len(comp_driven)/len(sig)*100:.1f}% of sig)')
        print(f'  → Per-cell-dominant: {len(percell_driven)} ({len(percell_driven)/len(sig)*100:.1f}% of sig)')
    else:
        print(f'  → No significant genes found.')

# Combine all
if all_decomp_results:
    full_decomp_df = pd.concat(all_decomp_results, ignore_index=True)
    print(f'\n==> Total decomposition records: {len(full_decomp_df)}')

    # Save complete results
    full_decomp_df.to_csv(f'{RESULTS_DIR}C10_full_decomposition_ALL_genes.csv', index=False)
    print(f'Saved to {RESULTS_DIR}C10_full_decomposition_ALL_genes.csv')
else:
    print("\n⚠️ No decomposition results were generated.")


Total tests to run: 173 genes × 6 lineages × 2 tissues × 6 comparisons = 12456
This may take 15-30 minutes on A100...


=== Comparison 1/6: NL→IT ===
  Progress: 2000/12456 (16.1%)
  Significant (p<0.05): 284
  → Composition-dominant: 1 (0.4% of sig)
  → Per-cell-dominant: 283 (99.6% of sig)

=== Comparison 2/6: NL→IA ===
  Progress: 4000/12456 (32.1%)
  Significant (p<0.05): 233
  → Composition-dominant: 12 (5.2% of sig)
  → Per-cell-dominant: 221 (94.8% of sig)

=== Comparison 3/6: IT→IA ===
  Progress: 6000/12456 (48.2%)
  Significant (p<0.05): 41
  → Composition-dominant: 3 (7.3% of sig)
  → Per-cell-dominant: 38 (92.7% of sig)

=== Comparison 4/6: NL→CR ===
  Progress: 8000/12456 (64.2%)
  Significant (p<0.05): 352
  → Composition-dominant: 19 (5.4% of sig)
  → Per-cell-dominant: 333 (94.6% of sig)

=== Comparison 5/6: NL→AR ===
  Progress: 10000/12456 (80.3%)
  Significant (p<0.05): 240
  → Composition-dominant: 16 (6.7% of sig)
  → Per-cell-dominant: 224 (93.3% of sig)

=== Comp

### Cell 8: Flag Composition-Confounded Claims

In [28]:
# ============================================================
# Cell 8: Flag composition-confounded claims
# For NL→IT specifically: identify genes where >50% of change
# is driven by subcluster composition, not per-cell expression
# ============================================================

nl_it_df = full_decomp_df[full_decomp_df['comparison'] == 'NL→IT'].copy()
sig_nl_it = nl_it_df[nl_it_df['p_value'] < 0.05].copy()

print('=== NL→IT: Composition vs Per-Cell Attribution ===')
print(f'Total significant gene×lineage×tissue combinations: {len(sig_nl_it)}')
print()

# ⚠️ DANGER ZONE: claims where composition drives >50%
danger = sig_nl_it[sig_nl_it['pct_composition'] > 50].sort_values('pct_composition', ascending=False)
safe = sig_nl_it[sig_nl_it['pct_percell'] > 50].sort_values('pct_percell', ascending=False)

print(f'⚠️ COMPOSITION-DOMINANT (>50% from subcluster shifts): {len(danger)}')
if len(danger) > 0:
    print(danger[['gene','tissue','lineage','pct_change','p_value',
                  'pct_composition','pct_percell','dominant_driver']].head(30).to_string())

print(f'\n✅ PER-CELL-DOMINANT (>50% genuine per-cell change): {len(safe)}')
if len(safe) > 0:
    print(safe[['gene','tissue','lineage','pct_change','p_value',
                'pct_composition','pct_percell','dominant_driver']].head(30).to_string())

# Specifically check 6-layer model genes
print('\n=== 6-Layer Model Gene Verification ===')
layer_genes = {
    'Layer1': ['TGFB1','LGALS9','LILRB1','SIGLEC10'],
    'Layer2': ['DNMT1','DNMT3A','TET2'],
    'Layer3': ['MTOR','LDHA','TFAM'],
    'Layer4': ['JAK1','STAT1','STAT2','STAT5A','SOCS1','SOCS3','TYK2'],
    'Layer5': ['TOX','TOX2','LAYN','CTLA4','TIGIT'],
    'Layer6': ['PRDM1','RORC']
}

for layer, genes in layer_genes.items():
    print(f'\n--- {layer} ---')
    for gene in genes:
        gene_rows = sig_nl_it[sig_nl_it['gene'] == gene]
        if len(gene_rows) == 0:
            # Check non-significant too
            gene_rows_ns = nl_it_df[nl_it_df['gene'] == gene]
            if len(gene_rows_ns) > 0:
                for _, row in gene_rows_ns.iterrows():
                    flag = '⚠️COMP' if row['pct_composition'] > 50 else '✅CELL'
                    print(f'  {gene} [{row["tissue"]}/{row["lineage"]}]: NS (p={row["p_value"]:.3f}) '
                          f'comp={row["pct_composition"]:.0f}% cell={row["pct_percell"]:.0f}% {flag}')
        else:
            for _, row in gene_rows.iterrows():
                flag = '⚠️COMP' if row['pct_composition'] > 50 else '✅CELL'
                print(f'  {gene} [{row["tissue"]}/{row["lineage"]}]: ★{row["pct_change"]:+.1f}% '
                      f'p={row["p_value"]:.4f} comp={row["pct_composition"]:.0f}% '
                      f'cell={row["pct_percell"]:.0f}% {flag}')

=== NL→IT: Composition vs Per-Cell Attribution ===
Total significant gene×lineage×tissue combinations: 284

⚠️ COMPOSITION-DOMINANT (>50% from subcluster shifts): 1
        gene tissue lineage  pct_change   p_value  pct_composition  pct_percell dominant_driver
173  HLA-DRA  Liver   CD4_T   95.597893  0.041126        56.012565    36.377755     composition

✅ PER-CELL-DOMINANT (>50% genuine per-cell change): 268
         gene tissue  lineage   pct_change   p_value  pct_composition  pct_percell dominant_driver
138      IRF1  Liver  Myeloid    70.131378  0.041126         0.051347    99.600027        per-cell
1126    CASP8  Blood  Myeloid   106.427979  0.031746         0.386206    99.516319        per-cell
235     PRDM1  Liver    CD4_T   -36.367371  0.002165         1.207527    98.225735        per-cell
1834      PKM  Blood        B    40.014236  0.031746         0.926741    97.916186        per-cell
1861     CD44  Blood        B    97.427620  0.007937         1.977009    97.297771        p

---
## PART C: All-Gene Discovery — Find Important 'Minor' Genes
### Cell 9: Complete ranking of ALL significant genes (not just top 10)

In [29]:
# ============================================================
# Cell 9: Comprehensive gene ranking — ALL genes, filtered for per-cell validity
# Two rankings:
#   (A) Traditional: by p-value (as in current V18)
#   (B) Per-cell-validated: by p-value BUT only genes where per-cell > 50%
# Compare: which genes appear in (A) but not (B)? → composition-confounded
#          which genes appear in (B) but were overlooked in top-10? → hidden gems
# ============================================================

nl_it_all = full_decomp_df[full_decomp_df['comparison'] == 'NL→IT'].copy()

for tissue in TISSUES:
    print(f'\n{"="*80}')
    print(f'  {tissue.upper()} — ALL Significant Genes (NL→IT)')
    print(f'{"="*80}')

    t_df = nl_it_all[nl_it_all['tissue'] == tissue]

    for lineage in LINEAGES:
        tl_df = t_df[t_df['lineage'] == lineage].copy()
        if len(tl_df) == 0:
            continue

        sig_tl = tl_df[tl_df['p_value'] < 0.05].sort_values('p_value')
        if len(sig_tl) == 0:
            print(f'\n  {lineage}: No significant genes at p<0.05')
            continue

        # Traditional top 10 vs all significant
        top10 = sig_tl.head(10)['gene'].tolist()
        all_sig = sig_tl['gene'].tolist()
        percell_valid = sig_tl[sig_tl['pct_percell'] > 50]['gene'].tolist()
        comp_confounded = sig_tl[sig_tl['pct_composition'] > 50]['gene'].tolist()

        # Hidden gems: significant + per-cell-valid but NOT in top 10
        hidden_gems = [g for g in percell_valid if g not in top10]

        # False positives: in top 10 but composition-confounded
        false_positives = [g for g in top10 if g in comp_confounded]

        print(f'\n  {lineage}: {len(all_sig)} significant genes total')
        print(f'    Top 10 by p-value: {top10}')
        print(f'    Per-cell validated (>50%): {len(percell_valid)}')
        print(f'    Composition-confounded (>50%): {len(comp_confounded)}')

        if false_positives:
            print(f'    ⚠️ TOP-10 FALSE POSITIVES (composition-driven): {false_positives}')

        if hidden_gems:
            print(f'    🔍 HIDDEN GEMS (valid but outside top 10): {hidden_gems[:20]}')
            # Show details of hidden gems
            for gem in hidden_gems[:10]:
                row = sig_tl[sig_tl['gene'] == gem].iloc[0]
                print(f'       → {gem}: {row["pct_change"]:+.1f}% p={row["p_value"]:.4f} '
                      f'percell={row["pct_percell"]:.0f}%')

print('\n==> All-gene analysis complete.')


  LIVER — ALL Significant Genes (NL→IT)

  Myeloid: 25 significant genes total
    Top 10 by p-value: ['HLA-DRA', 'HLA-DRB1', 'HLA-DPB1', 'HLA-DPA1', 'CD74', 'DNMT1', 'BAK1', 'HLA-DQB1', 'HDAC2', 'TOX']
    Per-cell validated (>50%): 22
    Composition-confounded (>50%): 0
    🔍 HIDDEN GEMS (valid but outside top 10): ['TAP1', 'CCR7', 'BATF', 'SMAD2', 'CASP3', 'NCR1', 'KLRF1', 'CD19', 'MTOR', 'IRF1', 'CASP8', 'MX1']
       → TAP1: +156.2% p=0.0087 percell=68%
       → CCR7: +460.6% p=0.0129 percell=53%
       → BATF: -62.4% p=0.0260 percell=57%
       → SMAD2: +59.6% p=0.0260 percell=81%
       → CASP3: +112.4% p=0.0260 percell=93%
       → NCR1: +99999.0% p=0.0284 percell=93%
       → KLRF1: +99999.0% p=0.0284 percell=61%
       → CD19: +99999.0% p=0.0284 percell=52%
       → MTOR: +327.3% p=0.0341 percell=81%
       → IRF1: +70.1% p=0.0411 percell=100%

  CD4_T: 27 significant genes total
    Top 10 by p-value: ['PRDM1', 'CD27', 'NCR3', 'CTLA4', 'TOX', 'LAYN', 'STAT2', 'TOX2', 'MTOR

### Cell 10: Subcluster-Matched Validation for Key Claims

In [30]:
# ============================================================
# Cell 10: Subcluster-matched validation
# For each significant gene: test expression within EACH subcluster
# This is the definitive test: if a gene is truly upregulated per-cell,
# it should be elevated within at least some subclusters, not just
# because the subcluster proportions changed.
# ============================================================

def subcluster_matched_test(adata, gene, tissue, lineage, group1='NL', group2='IT'):
    """Test gene expression within each subcluster separately."""
    mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_LINEAGE] == lineage)
    sub = adata[mask]

    if gene not in sub.var_names:
        return []

    gene_idx = list(sub.var_names).index(gene)
    expr = sub.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()

    obs = sub.obs.copy()
    obs['expr'] = expr

    results = []
    for sc_name in obs[COL_SUBCLUSTER].unique():
        sc_obs = obs[obs[COL_SUBCLUSTER] == sc_name]
        g1_donors = sc_obs[sc_obs[COL_DISEASE] == group1].groupby(COL_DONOR)['expr'].mean()
        g2_donors = sc_obs[sc_obs[COL_DISEASE] == group2].groupby(COL_DONOR)['expr'].mean()

        if len(g1_donors) < 2 or len(g2_donors) < 2:
            continue

        mean_g1 = g1_donors.mean()
        mean_g2 = g2_donors.mean()

        try:
            _, pval = mannwhitneyu(g1_donors.values, g2_donors.values, alternative='two-sided')
        except:
            pval = 1.0

        if mean_g1 > 0:
            pct = ((mean_g2 - mean_g1) / mean_g1) * 100
        else:
            pct = 99999 if mean_g2 > 0 else 0

        results.append({
            'gene': gene, 'tissue': tissue, 'lineage': lineage,
            'subcluster': sc_name,
            'mean_g1': mean_g1, 'mean_g2': mean_g2,
            'pct_change': pct, 'p_value': pval,
            'n_g1_donors': len(g1_donors), 'n_g2_donors': len(g2_donors),
            'direction': 'UP' if mean_g2 > mean_g1 else 'DOWN'
        })

    return results

# Run for ALL significant NL→IT genes (not just top 10!)
sig_genes_to_validate = nl_it_all[
    (nl_it_all['p_value'] < 0.05)
][['gene','tissue','lineage']].drop_duplicates()

print(f'Validating {len(sig_genes_to_validate)} significant gene×tissue×lineage combinations...')

sc_matched_results = []
for idx, (_, row) in enumerate(sig_genes_to_validate.iterrows()):
    res = subcluster_matched_test(adata, row['gene'], row['tissue'], row['lineage'])
    sc_matched_results.extend(res)
    if (idx + 1) % 100 == 0:
        print(f'  Progress: {idx+1}/{len(sig_genes_to_validate)}')

sc_matched_df = pd.DataFrame(sc_matched_results)
print(f'Total subcluster-matched tests: {len(sc_matched_df)}')

# Save
sc_matched_df.to_csv(f'{RESULTS_DIR}C10_subcluster_matched_validation.csv', index=False)
print(f'Saved to {RESULTS_DIR}')

Validating 284 significant gene×tissue×lineage combinations...
  Progress: 100/284
  Progress: 200/284
Total subcluster-matched tests: 2236
Saved to /content/drive/MyDrive/ITLAS/results/version18-analysis/C10_SimpsonsParadox/


### Cell 11: Final Verdict — Classify Each Gene's Validity

In [31]:
# ============================================================
# Cell 11: Final gene classification
# For each significant gene: classify validity based on decomposition
# + subcluster-matched evidence
# ============================================================

def classify_gene(gene, tissue, lineage, decomp_df, matched_df):
    """Classify a gene's reliability."""
    # Get decomposition result
    d_row = decomp_df[
        (decomp_df['gene'] == gene) &
        (decomp_df['tissue'] == tissue) &
        (decomp_df['lineage'] == lineage) &
        (decomp_df['comparison'] == 'NL→IT')
    ]
    if len(d_row) == 0:
        return 'NO_DATA'
    d_row = d_row.iloc[0]

    # Get subcluster-matched results
    m_rows = matched_df[
        (matched_df['gene'] == gene) &
        (matched_df['tissue'] == tissue) &
        (matched_df['lineage'] == lineage)
    ]

    # Count subclusters with consistent direction
    overall_dir = 'UP' if d_row['pct_change'] > 0 else 'DOWN'
    consistent_sc = len(m_rows[m_rows['direction'] == overall_dir])
    sig_consistent = len(m_rows[(m_rows['direction'] == overall_dir) & (m_rows['p_value'] < 0.1)])
    total_sc = len(m_rows)

    pct_percell = d_row['pct_percell']
    pct_comp = d_row['pct_composition']

    # Classification
    if pct_percell >= 70 and sig_consistent >= 1:
        return 'ROBUST'  # Strong per-cell signal confirmed at subcluster level
    elif pct_percell >= 50 and consistent_sc / max(total_sc, 1) >= 0.5:
        return 'VALID'   # Per-cell dominant with directional consistency
    elif pct_comp >= 70:
        return 'COMPOSITION_DRIVEN'  # Likely Simpson's paradox
    elif pct_comp >= 50:
        return 'MIXED_CAUTION'  # Ambiguous, needs caution
    else:
        return 'VALID'  # Per-cell dominant by default

# Classify all significant NL→IT genes
classifications = []
for _, row in sig_genes_to_validate.iterrows():
    d_data = full_decomp_df[
        (full_decomp_df['gene'] == row['gene']) &
        (full_decomp_df['tissue'] == row['tissue']) &
        (full_decomp_df['lineage'] == row['lineage']) &
        (full_decomp_df['comparison'] == 'NL→IT')
    ]
    if len(d_data) == 0:
        continue
    d_data = d_data.iloc[0]

    verdict = classify_gene(row['gene'], row['tissue'], row['lineage'],
                           full_decomp_df, sc_matched_df)

    classifications.append({
        'gene': row['gene'],
        'tissue': row['tissue'],
        'lineage': row['lineage'],
        'pct_change': d_data['pct_change'],
        'p_value': d_data['p_value'],
        'fdr_q': d_data['fdr_q'],
        'pct_composition': d_data['pct_composition'],
        'pct_percell': d_data['pct_percell'],
        'verdict': verdict
    })

verdict_df = pd.DataFrame(classifications)

# Summary
print('=== FINAL GENE CLASSIFICATION (NL→IT) ===')
print(verdict_df['verdict'].value_counts())
print()

# Flag any 6-layer genes that are composition-driven
print('=== 6-LAYER MODEL GENES: VERDICT ===')
all_layer_genes = [g for genes in layer_genes.values() for g in genes]
layer_verdicts = verdict_df[verdict_df['gene'].isin(all_layer_genes)].sort_values(['gene','tissue'])
if len(layer_verdicts) > 0:
    print(layer_verdicts[['gene','tissue','lineage','pct_change','p_value',
                          'pct_composition','pct_percell','verdict']].to_string())
else:
    print('No 6-layer genes in significant set (check column name matching)')

# Save final verdicts
verdict_df.to_csv(f'{RESULTS_DIR}C10_gene_verdicts_NLvsIT.csv', index=False)

# Also save the COMPOSITION_DRIVEN and MIXED_CAUTION as a separate warning file
warnings_df = verdict_df[verdict_df['verdict'].isin(['COMPOSITION_DRIVEN', 'MIXED_CAUTION'])]
warnings_df.to_csv(f'{RESULTS_DIR}C10_WARNINGS_composition_confounded.csv', index=False)
print(f'\n⚠️ {len(warnings_df)} gene claims flagged for composition confounding.')
print(f'Saved to {RESULTS_DIR}')

=== FINAL GENE CLASSIFICATION (NL→IT) ===
verdict
ROBUST           194
VALID             89
MIXED_CAUTION      1
Name: count, dtype: int64

=== 6-LAYER MODEL GENES: VERDICT ===
         gene tissue  lineage    pct_change   p_value  pct_composition  pct_percell verdict
200     CTLA4  Blood    CD8_T    -71.505180  0.007937         1.420374    96.531639   VALID
34      CTLA4  Liver    CD4_T    171.127182  0.004329        15.088900    69.470838   VALID
141     DNMT1  Blood  Myeloid    163.816528  0.007937         3.221838    95.992020  ROBUST
215     DNMT1  Blood       NK     64.297264  0.031746         1.678232    96.674335  ROBUST
9       DNMT1  Liver  Myeloid    125.223198  0.002165         3.583023    70.057927  ROBUST
142    DNMT3A  Blood  Myeloid    348.297607  0.011925         2.244333    87.985687  ROBUST
182    DNMT3A  Blood    CD4_T    133.234695  0.007937         2.130770    75.648079  ROBUST
199    DNMT3A  Blood    CD8_T     90.336815  0.015873         4.026584    77.233035  RO

### Cell 12: Run Same Analysis for IT→IA and NL→CR (Result 4)

In [32]:
# ============================================================
# Cell 12: Repeat verdict for IT→IA transition and CR scar genes
# ============================================================

for comp_name in ['IT→IA', 'NL→CR', 'NL→AR', 'IA→AR']:
    comp_df = full_decomp_df[full_decomp_df['comparison'] == comp_name].copy()
    sig = comp_df[comp_df['p_value'] < 0.05]

    if len(sig) == 0:
        print(f'\n{comp_name}: No significant genes.')
        continue

    comp_driven = sig[sig['pct_composition'] > 50]
    percell_driven = sig[sig['pct_percell'] > 50]

    print(f'\n{"="*60}')
    print(f'  {comp_name}: {len(sig)} significant genes')
    print(f'  Composition-dominant: {len(comp_driven)} ({len(comp_driven)/len(sig)*100:.1f}%)')
    print(f'  Per-cell-dominant: {len(percell_driven)} ({len(percell_driven)/len(sig)*100:.1f}%)')
    print(f'{"="*60}')

    # Show composition-confounded genes (potential Simpson's paradox)
    if len(comp_driven) > 0:
        print(f'\n  ⚠️ Composition-confounded genes:')
        print(comp_driven[['gene','tissue','lineage','pct_change','p_value',
                          'pct_composition','pct_percell']].sort_values('pct_composition',
                          ascending=False).head(20).to_string())

    # Save
    safe_name = comp_name.replace('→', '_to_')
    sig.to_csv(f'{RESULTS_DIR}C10_decomposition_{safe_name}_significant.csv', index=False)


  IT→IA: 41 significant genes
  Composition-dominant: 1 (2.4%)
  Per-cell-dominant: 34 (82.9%)

  ⚠️ Composition-confounded genes:
      gene tissue lineage  pct_change   p_value  pct_composition  pct_percell
4557  RORC  Liver   CD8_T  -58.426666  0.017316        91.716905     3.737055

  NL→CR: 352 significant genes
  Composition-dominant: 6 (1.7%)
  Per-cell-dominant: 286 (81.2%)

  ⚠️ Composition-confounded genes:
      gene tissue  lineage  pct_change   p_value  pct_composition  pct_percell
7913  CCR7  Blood       NK  141.408844  0.035714        79.299139    15.651938
6699  IL7R  Liver    CD8_T  -46.284523  0.047619        62.381838    32.598802
7385  IDO1  Blood  Myeloid  556.205017  0.026179        57.446502    18.696629
7910  IL7R  Blood       NK  155.640228  0.035714        52.285481    43.765809
6647  GZMH  Liver    CD8_T  115.450645  0.023810        52.019614    44.989275
6425  TSC1  Liver    CD4_T   53.856712  0.047619        50.598654     8.095614

  NL→AR: 240 significant

---
## PART D: Summary Report Generation
### Cell 13: Generate comprehensive summary

In [33]:
# ============================================================
# Cell 13: Summary report
# ============================================================

print('='*80)
print('  C10 SIMPSON\'S PARADOX VERIFICATION — FINAL REPORT')
print('='*80)

# Part A: Subcluster composition shifts
print('\n[PART A] Subcluster Composition Shifts (NL→IT)')
sc_nl_it = sc_change_dfs.get('NL_IT')
if sc_nl_it is not None:
    sig_sc = sc_nl_it[sc_nl_it['p_value'] < 0.05]
    print(f'  Significant subcluster proportion changes: {len(sig_sc)}')
    if len(sig_sc) > 0:
        for _, row in sig_sc.iterrows():
            print(f'    {row["tissue"]}/{row["lineage"]}/{row["subcluster"]}: '
                  f'{row["pct_change"]:+.1f}% (p={row["p_value"]:.4f})')

# Part B: Decomposition summary
print('\n[PART B] Expression Decomposition (NL→IT)')
nl_it_sig = full_decomp_df[
    (full_decomp_df['comparison'] == 'NL→IT') &
    (full_decomp_df['p_value'] < 0.05)
]
print(f'  Total significant: {len(nl_it_sig)}')
print(f'  Per-cell dominant: {len(nl_it_sig[nl_it_sig["pct_percell"] > 50])} '
      f'({len(nl_it_sig[nl_it_sig["pct_percell"] > 50])/max(len(nl_it_sig),1)*100:.1f}%)')
print(f'  Composition dominant: {len(nl_it_sig[nl_it_sig["pct_composition"] > 50])} '
      f'({len(nl_it_sig[nl_it_sig["pct_composition"] > 50])/max(len(nl_it_sig),1)*100:.1f}%)')

# Part C: All-gene discovery
print('\n[PART C] All-Gene Discovery vs Top-10 Comparison')
for tissue in TISSUES:
    for lineage in LINEAGES:
        tl = nl_it_sig[
            (nl_it_sig['tissue'] == tissue) &
            (nl_it_sig['lineage'] == lineage)
        ].sort_values('p_value')
        if len(tl) <= 10:
            continue

        top10 = set(tl.head(10)['gene'])
        valid_beyond_top10 = tl.iloc[10:][
            tl.iloc[10:]['pct_percell'] > 50
        ]
        if len(valid_beyond_top10) > 0:
            print(f'  {tissue}/{lineage}: {len(valid_beyond_top10)} valid genes beyond top 10')
            for _, row in valid_beyond_top10.head(5).iterrows():
                print(f'    → {row["gene"]}: {row["pct_change"]:+.1f}% p={row["p_value"]:.4f} '
                      f'percell={row["pct_percell"]:.0f}%')

# Part D: 6-Layer model verification
print('\n[PART D] 6-Layer Model Gene Status')
if len(verdict_df) > 0:
    for layer, genes in layer_genes.items():
        layer_v = verdict_df[verdict_df['gene'].isin(genes)]
        if len(layer_v) == 0:
            print(f'  {layer}: No significant genes in verdict set')
        else:
            robust = len(layer_v[layer_v['verdict'] == 'ROBUST'])
            valid = len(layer_v[layer_v['verdict'] == 'VALID'])
            danger = len(layer_v[layer_v['verdict'].isin(['COMPOSITION_DRIVEN', 'MIXED_CAUTION'])])
            print(f'  {layer}: ✅ROBUST={robust} ✅VALID={valid} ⚠️CAUTION={danger}')

print('\n[FILES GENERATED]')
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith('C10_'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f))
        print(f'  {f} ({size/1024:.1f} KB)')

print('\n==> C10 Analysis Complete.')

  C10 SIMPSON'S PARADOX VERIFICATION — FINAL REPORT

[PART A] Subcluster Composition Shifts (NL→IT)
  Significant subcluster proportion changes: 10
    Blood/CD8_T/g9d2T-c01-CD27: -76.3% (p=0.0473)
    Blood/Myeloid/plasmaB_c02-CD52: +42.6% (p=0.0117)
    Blood/Myeloid/mono_c01-CD14: -72.9% (p=0.0117)
    Blood/NK/CD8T_c03-CX3CR1: +2415.8% (p=0.0256)
    Liver/B/cDC2-CLEC10A: +99999.0% (p=0.0484)
    Liver/Myeloid/mono_c01-CD14: -92.5% (p=0.0048)
    Liver/Myeloid/cDC2-CLEC10A: +378.9% (p=0.0295)
    Liver/NK/CD4T_c08-GZMK: -56.9% (p=0.0043)
    Liver/NK/plasmaB_c02-CD52: -9.1% (p=0.0411)
    Liver/gdT/CD4T_c07-TIMP1: -78.6% (p=0.0275)

[PART B] Expression Decomposition (NL→IT)
  Total significant: 284
  Per-cell dominant: 268 (94.4%)
  Composition dominant: 1 (0.4%)

[PART C] All-Gene Discovery vs Top-10 Comparison
  Liver/Myeloid: 12 valid genes beyond top 10
    → TAP1: +156.2% p=0.0087 percell=68%
    → CCR7: +460.6% p=0.0129 percell=53%
    → BATF: -62.4% p=0.0260 percell=57%
    